# CafChem notebook: fine-tuning a MACE materials model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/UMA_calcs/blob/main/mace_training_archive/TrainMace_CafChem.ipynb)

This notebook takes **your own reference data** (energies and forces), fine-tunes the
**MACE-MP-0** foundation model (`"small"`) on it — with a replay head that keeps the
foundation's general chemistry — then runs a short molecular-dynamics
simulation **before** and **after** fine-tuning so you can see the difference: it plots
energy per atom and temperature against time for both models, and shows the trajectories.

Getting started:
- Select a GPU runtime: `Runtime ▸ Change runtime type ▸ T4 GPU`
- On a T4, fine-tuning a few hundred configurations takes on the order of 10-30 minutes,
  depending on `MAX_NUM_EPOCHS` and the data set size


## What you need to provide

**1. A fine-tuning file** (required) — an extended-XYZ file written by ASE (say, `train.xyz`),
containing one configuration per frame, where each frame carries:

- an energy in its `info` field — a key such as `energy`, `energy_xtb` or `REF_energy`
- forces in its `arrays` field — a key such as `forces`, `forces_xtb` or `REF_forces`

If your frames were calculated with ASE (e.g. with an XTB or VASP/QE calculator attached),
building the file looks like this:

```python
from ase.io import read, write

frames = read('raw_data.xyz', ':')
for at in frames:
    at.info['energy']   = at.get_potential_energy()   # eV
    at.arrays['forces'] = at.get_forces()             # eV/Angstrom
write('train.xyz', frames)
```

Exactly which key names your data uses does not need guessing: the upload cell below
prints every key it finds in your file and **auto-picks** the most likely pair
(`energy`/`REF_energy`/`energy_xtb` for energies, `forces`/`REF_forces`/`forces_xtb` for
forces), which the Settings cell then uses — change the two lines there if the pick is wrong.

**2. A starting structure** (optional, `start.xyz`) — the configuration the before/after
MD runs start from. If you do not upload one, the first configuration of your training
file is used.

Both files are uploaded in the **"Upload your data"** cell, which pops up a file picker.
If you don't upload anything, the notebook falls back to a bundled example data set
(xtb energies/forces for a small solvent molecule) so you can still run the whole flow.


## Install

In [ ]:
# quick check that a GPU is attached (Runtime > Change runtime type > T4 GPU)
!nvidia-smi -L


In [ ]:
# If a torch/runtime error shows up further down, do
#   Runtime > Restart session
# then re-run the two install cells (this one and the next code cell) before carrying on.
!pip install -q mace-torch py3dmol
!pip install -q git+https://github.com/imagdau/aseMolec@3b1667abd2e4b061aa9f2e910634d38e17fe4779
!pip install -q cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12


## Setup

This cell pulls the helper code (`mace_train.py`: `train_mace`, `make_train_file`,
`simpleMD`, `view_traj`) plus the example data and config templates from the repo
archive, then imports them. It also moves the working directory (the notebook keeps
everything it creates under `mace_training_archive`).


In [ ]:
import os, sys

if not os.path.isdir('UMA_calcs'):
    os.system('git clone -q https://github.com/MauricioCafiero/UMA_calcs.git')
if os.getcwd().split('/')[-1] != 'mace_training_archive':
    os.chdir(os.path.join('UMA_calcs', 'mace_training_archive'))
if 'code' not in sys.path:
    sys.path.append('code')

from mace_train import train_mace, make_train_file, simpleMD, view_traj
from ase.io import read, write
import numpy as np
import matplotlib.pyplot as plt

print('working directory:', os.getcwd())


## Upload your data

Run this cell and pick your `train.xyz` (and, if you have one, `start.xyz`) in the
file picker. On a re-run with nothing selected, the bundled example data is used.


In [ ]:
from google.colab import files

TRAIN_FILE = None   # e.g. TRAIN_FILE = 'train.xyz'    (set by hand if not on Colab)
START_FILE = None   # e.g. START_FILE = 'start.xyz'    (optional)

uploaded = files.upload()   # pick your train.xyz (and optionally start.xyz)
xyz_files  = [n for n in uploaded if n.lower().endswith('.xyz')]
start_list = [n for n in xyz_files if 'start' in n.lower()]
train_list = [n for n in xyz_files if n not in start_list]

if train_list:
    TRAIN_FILE = train_list[0]
    if len(train_list) > 1:
        print('Several candidate training files uploaded; using', TRAIN_FILE,
              '- edit this cell to pick another.')
    if start_list:
        START_FILE = start_list[0]

if TRAIN_FILE is None:
    TRAIN_FILE = 'data/solvent_xtb.xyz'
    print('No training file uploaded - using the bundled example data.')

frames = read(TRAIN_FILE, ':')
info_keys = sorted({k for at in frames for k in at.info})
array_keys = sorted({k for at in frames for k in at.arrays})

# pick the keys this file most likely uses for energies and forces
def auto_pick(available, candidates, prefix):
    for c in candidates:
        if c in available:
            return c
    for k in available:
        if prefix in k:
            return k

E_KEY_DEFAULT = auto_pick(info_keys, ['energy', 'REF_energy', 'energy_xtb'], 'energy')
F_KEY_DEFAULT = auto_pick(array_keys, ['forces', 'REF_forces', 'forces_xtb'], 'forc')

print(f'{TRAIN_FILE}: {len(frames)} configurations')
print(' energy keys available:', info_keys, '-> auto-picked:', E_KEY_DEFAULT)
print(' force keys available :', array_keys, '-> auto-picked:', F_KEY_DEFAULT)


## Settings

Edit the block in the next cell before running it:

- `ENERGY_KEY` / `FORCES_KEY` — auto-picked from your uploaded file (see the upload cell
  output; the defaults are what ASE writes, `energy` / `forces`). Change these two only
  if the auto-pick chose the wrong pair.
- `NAME` — the output model name; `SEED` — controls the data split and the run id
- `MAX_NUM_EPOCHS` — 500 is a sensible default; set it lower (e.g. 200) for a quick run,
  higher if the validation errors are still improving at the end
- `BATCH_SIZE` — lower this if the GPU runs out of memory
- `MD_TEMP` / `MD_STEPS` — temperature (K) and length (1 fs steps) of the before/after MD.
  1200 K is aggressive on purpose (it exposes model failures); for bulk/periodic systems
  or fragile molecules, start lower.


In [ ]:
# ==================== edit these ====================
ENERGY_KEY     = E_KEY_DEFAULT   # auto-picked from your file; change by hand if it picked wrong
FORCES_KEY     = F_KEY_DEFAULT   # auto-picked from your file
NAME           = 'my_model'
SEED           = 42
MAX_NUM_EPOCHS = 500
BATCH_SIZE     = 10
MD_TEMP        = 1200       # K
MD_STEPS       = 2000       # number of 1 fs steps
MD_SAVE_EVERY  = 10         # save a frame every MD_SAVE_EVERY fs
DEVICE         = 'cuda'     # 'cuda' on a Colab GPU, 'cpu' otherwise
# ====================================================

# sanity-check the keys against the uploaded data
have_energy = any(ENERGY_KEY in at.info for at in frames)
have_forces = any(FORCES_KEY in at.arrays for at in frames)
if not (have_energy and have_forces):
    raise ValueError(
        f'Keys not found: energy {ENERGY_KEY} present: {have_energy}, '
        f'forces {FORCES_KEY} present: {have_forces}. '
        'Pick keys from the lists printed by the upload cell, '
        'or fix the file so it carries energies and forces.'
    )

# seeded shuffle + 80/10/10 train/valid/test split
rng = np.random.default_rng(SEED)
order = rng.permutation(len(frames))
n_test = max(1, len(frames) // 10)
n_val = max(1, len(frames) // 10)
if len(frames) - n_test - n_val < 3:
    raise RuntimeError(
        f'Only {len(frames)} configurations - fine-tuning needs more data '
        f'(at least 3 spare after the ~10% valid/test splits).'
    )
write('data/student_test.xyz',  [frames[i] for i in order[:n_test]])
write('data/student_valid.xyz', [frames[i] for i in order[n_test:n_test + n_val]])
write('data/student_train.xyz', [frames[i] for i in order[n_test + n_val:]])
print(f'split: {len(frames) - n_test - n_val} train / {n_val} valid / {n_test} test configurations')

# write the fine-tuning config, then set the device from DEVICE above
# (the templates fix it to cuda for Colab; rewrite the line so cpu runs also work)
CONFIG = 'config/student.yml'
make_train_file(
    filename=CONFIG,
    ft=True,
    name=NAME,
    seed=SEED,
    max_num_epochs=MAX_NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    energy_key=ENERGY_KEY,
    forces_key=FORCES_KEY,
    train_file='data/student_train.xyz',
    valid_file='data/student_valid.xyz',
    test_file='data/student_test.xyz',
)
with open(CONFIG) as f:
    yaml_text = f.read()
with open(CONFIG, 'w') as f:
    f.write(yaml_text.replace('device: cuda', f'device: {DEVICE}'))
print(open(CONFIG).read())


## Fine-tune MACE-MP-0 on your data

This runs MACE's fine-tuning of the `small` MACE-MP-0 foundation model **with a replay
head** (`multiheads_finetuning`): one head fits your data while a second head replays
300 Materials Project configurations from the foundation's own training set, so your
chemistry is learned without the model forgetting its general one. MACE supplies the
replay data automatically for this foundation model; the price of this protection is
that the mode fixes the learning rate to 1e-4 with EMA on top. Expect 10-30 minutes on
a T4 for a few hundred configurations. Training losses and the final test RMSEs (energy
and forces on the held-out test set) are printed below.


In [ ]:
import glob, os

os.makedirs('MACE_models', exist_ok=True)
for f in glob.glob('MACE_models/*.pt'):   # stale checkpoints break re-runs with a new data set
    os.remove(f)

train_mace(CONFIG)

# final model name differs between MACE versions ({name}_stagetwo.model vs
# {name}_run-{seed}_stagetwo.model), so find it rather than guess
cands = [p for p in glob.glob(f'MACE_models/{NAME}*model') if '_compiled' not in p]
staged = [p for p in cands if 'stagetwo' in p]
MODEL_PATH = sorted(staged)[0] if staged else sorted(cands)[-1]
print('fine-tuned model:', MODEL_PATH)


## Dynamics before and after fine-tuning

Both runs use the same starting structure, temperature and length, so the only difference
is the potential: first the un-fine-tuned MACE-MP-0, then your fine-tuned model.


In [ ]:
# MD with the base foundation model (before fine-tuning)
from mace.calculators import mace_mp

base_model = mace_mp(model='small', device=DEVICE)

# starting structure: your uploaded start.xyz, else the first configuration of the data
if START_FILE is not None:
    start = read(START_FILE)
else:
    start = frames[0].copy()
start.calc = None

simpleMD(start, temp=MD_TEMP, calc=base_model,
         fname='moldyn/base_md.xyz', s=MD_SAVE_EVERY, T=MD_STEPS)


In [ ]:
# MD with the fine-tuned model
from mace.calculators import MACECalculator

ft_model = MACECalculator(model_paths=[MODEL_PATH], device=DEVICE, enable_cueq=False)

simpleMD(start, temp=MD_TEMP, calc=ft_model,
         fname='moldyn/ft_md.xyz', s=MD_SAVE_EVERY, T=MD_STEPS)


In [ ]:
# before/after summary plot
base_traj = read('moldyn/base_md.xyz', ':')
ft_traj = read('moldyn/ft_md.xyz', ':')

def times(traj):
    return np.arange(len(traj)) * MD_SAVE_EVERY

def energies(traj):
    return [at.info['energy_mace'] / len(at) for at in traj]

def temps(traj):
    if 'temperature_mace' in base_traj[0].info:      # stored by newer mace_train.py
        return [at.info['temperature_mace'] for at in traj]
    return [at.get_temperature() for at in traj]     # falls back to the saved momenta

fig, ax = plt.subplots(2, 1, figsize=(6, 8), sharex='all', gridspec_kw={'hspace': 0.05})
ax[0].plot(times(base_traj), energies(base_traj), color='b', label='MACE-MP-0 (before)')
ax[0].plot(times(ft_traj), energies(ft_traj), color='r', label=f'{NAME} (after)')
ax[0].set_ylabel('E (eV/atom)')
ax[0].legend()
ax[1].plot(times(base_traj), temps(base_traj), color='b', label='MACE-MP-0 (before)')
ax[1].plot(times(ft_traj), temps(ft_traj), color='r', label=f'{NAME} (after)')
ax[1].set_ylabel('T (K)')
ax[1].set_xlabel('Time (fs)')
fig.savefig('moldyn/before_after_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
view_traj('moldyn/base_md.xyz')   # before
view_traj('moldyn/ft_md.xyz')    # after


## Results and reuse

Files produced in `mace_training_archive/`:

- `MACE_models/{NAME}_stagetwo.model` (or `{NAME}_run-{SEED}_stagetwo.model` on older
  MACE versions) — the fine-tuned MACE model. It carries two heads — `pt_head` (the
  Materials Project replay for MACE-MP-0) and `Default` (your fine-tuned one);
  `MACECalculator` selects `Default` automatically, as in the cell above, so look for
  the message `Using head Default out of [...]`. Reuse it anywhere as an ASE calculator:
  ```python
  from mace.calculators import MACECalculator
  calc = MACECalculator(model_paths=['path/to/model_stagetwo.model'], device='cuda')
  ```
- `moldyn/base_md.xyz`, `moldyn/ft_md.xyz` — the before/after MD trajectories
- `moldyn/before_after_dynamics.png` — the comparison plot
- `config/student.yml` and `data/student_*.xyz` — the generated config and data splits
  (re-run the Settings cell after changing anything, then the training cell again)


In [ ]:
from google.colab import files

files.download(MODEL_PATH)                        # the fine-tuned model
files.download('moldyn/before_after_dynamics.png')  # the comparison plot
